In [1]:
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
# import xgboost as xgb

from glob import glob
# import psi4
# from helper_CC_ML_spacial import *

import pyscf
from pyscf import gto, scf, mcscf, cc

import ffsim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorSampler, BitArray
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import bit_array_to_arrays


from ansatzmap import get_zigzag_physical_layout
from math import comb
from tqdm import tqdm

from DDLUCJ import DDLUCJ, GrabAmps


In [2]:
BasisDirs=glob('data/*')

In [3]:
energyDF=pd.read_csv("../../../classical/energies.csv",index_col=0)

In [4]:
energyDF

,Method,Energy,Name,Basis Set
0,HF,-74.960552,water,STO-3G
1,CASCI,-75.009479,water,STO-3G
2,SCI,-75.009478,water,STO-3G
0,HF,-55.454318,ammonia,STO-3G
1,CASCI,-55.519929,ammonia,STO-3G
...,...,...,...,...
1,SCI,-191.953772,GDB04_33,aug-cc-pVDZ
0,HF,-215.949690,GDB04_65,aug-cc-pVDZ
1,SCI,-215.950412,GDB04_65,aug-cc-pVDZ
0,HF,-336.812790,GDB04_5,aug-cc-pVDZ


In [5]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [6]:
BasisSets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

In [9]:
dfrerun = pd.read_excel("repostprocess.xlsx",index_col=0)

In [ ]:
for _, row in list(dfrerun.iterrows())[0:1]:
    basisrerun, namererun, method, layer, injected, ref_e, natom, deviation = row.to_list()
    filename = f"./jobids/{namererun}_LUCJ_L{layer}_{basisrerun}_{injected}.txt"
    
    with open(filename,'r') as f:
        name,basis,k,L,JobID = [j.strip() for j in f.readlines()]
    print(name,basis,k,L,JobID)
    moldict = moldf[moldf['molecule']==name]

    n_electrons=moldict['n_electrons'].values[0]
    num_orbitals=moldict['num_orbitals'].values[0]
    xyzname = moldict['mol_filename'].values[0]
    
    pathxyz = os.path.join(os.path.expanduser("~"),"DDLUCJ/classical/structures/",xyzname)



    JobPath = f"./jobids/{name}_LUCJ_L{L}_{basis}_{k}.txt" 
    EnergyPath = f"./energies/{name}_LUCJ_L{L}_{basis}_{k}.txt" 

    # if os.path.exists(JobPath)==True and os.path.exists(EnergyPath)==False:
    print(f"Running {name}_LUCJ_L{L}_{basis}_{k}")
    # run(pathxyz,name,basis,n_electrons,num_orbitals,L,k)
    ampdict = GrabAmps(f"{name}",f"{basis}")
    
    t1, t2 = ampdict[f"{k}"]
    print(comb(num_orbitals,n_electrons//2)**2)

    initDDLUCJ = DDLUCJ(StructurePath=f"{pathxyz}", 
                        BasisSet=f"{basis}", 
                        NElec=int(n_electrons),
                        NOrb=int(num_orbitals),
                        injected=True,
                        t1=t1, 
                        t2=t2,
                        n_reps = int(L),
                        optimization_level=3,
                        temp_dir="./",
                        clean_temp_dir=True,
                        n_jobs=8, 
                        num_batches = 10,
                        max_iterations=5,
                        samples_per_batch=5000,
                        verbose=False)
    
    counts = np.load(f"./counts/{name}_LUCJ_L{L}_{basis}_{k}.npz")
    bitstrings = counts['bitstrings']
    probarr = counts['probarr']
    bitstrings = BitArray.from_bool_array(bitstrings)
    
    result_history, result = initDDLUCJ(postprocess=True,BitArray=bitstrings)     
    new_energy = result.energy + initDDLUCJ.nuclear_repulsion_energy    

(Z)-1-fluoroprop-1-ene STO-3G CCSD 1 d3lvg6j4kkus739d5sug
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_CCSD
4173746850625
converged SCF energy = -213.116469454496
Iteration 1
	Subsample 0
		Energy: -209.62913208972117
		Subspace dimension: 70756
	Subsample 1
		Energy: -208.15479582843415
		Subspace dimension: 70756
	Subsample 2
		Energy: -208.15479551237382
		Subspace dimension: 70756
	Subsample 3
		Energy: -209.6291320897214
		Subspace dimension: 70756
	Subsample 4
		Energy: -209.62913208971895
		Subspace dimension: 70756
	Subsample 5
		Energy: -209.62913208972878
		Subspace dimension: 70756
	Subsample 6
		Energy: -208.154795607101
		Subspace dimension: 70756
	Subsample 7
		Energy: -209.62913208972674
		Subspace dimension: 70756
	Subsample 8
		Energy: -208.15479575004775
		Subspace dimension: 70756
	Subsample 9
		Energy: -209.62913208972532
		Subspace dimension: 70756


In [ ]:

energyDF[(energyDF['Method']=='SCI')&(energyDF["Name"]=="GDB04_65")]

In [ ]:
# os.mkdir('energies')

In [ ]:
postprocessed = []
for i in tqdm(sorted(glob("./jobids/*txt")),desc='Running'):
    with open(i,'r') as f:
        name,basis,k,L,JobID = [j.strip() for j in f.readlines()]
    print(name,basis,k,L,JobID)
    moldict = moldf[moldf['molecule']==name]

    n_electrons=moldict['n_electrons'].values[0]
    num_orbitals=moldict['num_orbitals'].values[0]
    xyzname = moldict['mol_filename'].values[0]
    
    pathxyz = os.path.join(os.path.expanduser("~"),"DDLUCJ/classical/structures/",xyzname)



    JobPath = f"./jobids/{name}_LUCJ_L{L}_{basis}_{k}.txt" 
    EnergyPath = f"./energies/{name}_LUCJ_L{L}_{basis}_{k}.txt" 

    # if os.path.exists(JobPath)==True and os.path.exists(EnergyPath)==False:
    print(f"Running {name}_LUCJ_L{L}_{basis}_{k}")
    # run(pathxyz,name,basis,n_electrons,num_orbitals,L,k)
    ampdict = GrabAmps(f"{name}",f"{basis}")
    
    t1, t2 = ampdict[f"{k}"]
    print(comb(num_orbitals,n_electrons//2)**2)

    initDDLUCJ = DDLUCJ(StructurePath=f"{pathxyz}", 
                        BasisSet=f"{basis}", 
                        NElec=int(n_electrons),
                        NOrb=int(num_orbitals),
                        injected=True,
                        t1=t1, 
                        t2=t2,
                        n_reps = int(L),
                        optimization_level=3,
                        temp_dir="./",
                        clean_temp_dir=True,
                        n_jobs=8, 
                        num_batches = 10,
                        max_iterations=5,
                        samples_per_batch=1000,
                        verbose=False)
    
    counts = np.load(f"./counts/{name}_LUCJ_L{L}_{basis}_{k}.npz")
    bitstrings = counts['bitstrings']
    probarr = counts['probarr']
    bitstrings = BitArray.from_bool_array(bitstrings)
    
    result_history, result = initDDLUCJ(postprocess=True,BitArray=bitstrings)     
    new_energy = result.energy + initDDLUCJ.nuclear_repulsion_energy
    EnergyPath = f"./energies/{name}_LUCJ_L{L}_{basis}_{k}_NB10_MI5_spb20_000.txt" 
    with open(EnergyPath,'w') as f:
        new_row={"Basis Set": f"{basis}", "Molecule": f"{name}", "Method": f"LUCJ(L={L})/{k}", "Energy": new_energy}
        for k,v in new_row.items():
            f.write(f"{v}\n")

In [ ]:
num_orbitals,n_electrons